# ⚡ Módulo 12 - Notebook 01: PySpark transformación avanzada

## 🔧 Manipulación y transformación de datos distribuidos

**Libro:** Saliendo de lo Pandito  
**Módulo:** 12 - PySpark Transformación Avanzada  
**Duración estimada:** 75 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Aplicar** transformaciones complejas con withColumn()  
✅ **Usar** pyspark.sql.functions para operaciones avanzadas  
✅ **Crear** columnas derivadas y calculadas  
✅ **Manejar** tipos de datos y conversiones  
✅ **Optimizar** transformaciones distribuidas

---

## 📋 Pre-requisitos

* ✅ Módulo 11 completado (PySpark Core)
* ✅ Conocimiento de Spark DataFrames
* ✅ Familiaridad con transformaciones lazy

---

## 📚 Contenido

1. withColumn() y Transformaciones de Columnas
2. pyspark.sql.functions: La Biblioteca Esencial
3. Operaciones Condicionales (when/otherwise)
4. Manejo de Valores Nulos
5. Conversión de Tipos de Datos
6. Caso Integrador: ETL Completo con Transformaciones

---

## 💡 Por qué importa

**Transformaciones son el corazón de ETL:**

* 🔧 **Limpieza:** Normalizar, validar, corregir datos
* 📊 **Enriquecimiento:** Agregar columnas calculadas
* 🔄 **Conversiones:** Cambiar tipos y formatos
* ⚡ **Optimización:** Aprovecchar paralelismo

**El 80% del tiempo en proyectos de datos**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🏛️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    print(f"\n📋 Esquema original:")
    df.printSchema()
    
    print(f"\n🎯 Este notebook aplicará transformaciones avanzadas:")
    print(f"   • Agregar columnas calculadas")
    print(f"   • Transformar tipos de datos")
    print(f"   • Limpiar y validar datos")
    print(f"   • Enriquecer con lógica de negocio")
    
    print(f"\n💡 Todas las transformaciones son LAZY (no ejecutan hasta acción)")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Transformaciones Avanzadas en PySpark

### 🔧 withColumn(): Añadir o Modificar Columnas

**Sintaxis básica:**
```python
df_nuevo = df.withColumn("nueva_col", expresion)
```

**Características:**
* Crea un NUEVO DataFrame (inmutable)
* Lazy (no ejecuta hasta acción)
* Puede agregar columna nueva o sobrescribir existente

**Ejemplos:**

**1️⃣ Columna nueva simple:**
```python
df = df.withColumn("descuento", F.lit(0.10))  # 10% fijo
```

**2️⃣ Columna calculada:**
```python
df = df.withColumn("precio_final", F.col("precio") * 0.9)
```

**3️⃣ Columna condicional:**
```python
df = df.withColumn(
    "categoria",
    F.when(F.col("ventas") > 100000, "Alto")
     .when(F.col("ventas") > 50000, "Medio")
     .otherwise("Bajo")
)
```

---

### 📦 pyspark.sql.functions: La Biblioteca Esencial

**Import estándar:**
```python
from pyspark.sql import functions as F
```

**Funciones más usadas:**

#### **Matemáticas:**
```python
F.col("ventas") + 100        # Suma
F.col("precio") * 1.21       # IVA
F.round(F.col("monto"), 2)   # Redondear
F.abs(F.col("diferencia"))   # Absoluto
```

#### **Strings:**
```python
F.upper(F.col("nombre"))           # Mayúsculas
F.lower(F.col("ciudad"))           # Minúsculas
F.trim(F.col("texto"))             # Quitar espacios
F.concat(F.col("a"), F.lit("-"), F.col("b"))  # Concatenar
F.substring(F.col("codigo"), 1, 3) # Subcadena
```

#### **Fechas:**
```python
F.current_date()                    # Fecha actual
F.year(F.col("fecha"))              # Extraer año
F.month(F.col("fecha"))             # Extraer mes
F.datediff(F.col("fecha2"), F.col("fecha1"))  # Días entre fechas
```

#### **Agregaciones:**
```python
F.sum(F.col("ventas"))
F.avg(F.col("precio"))
F.count(F.col("id"))
F.min(F.col("fecha"))
F.max(F.col("monto"))
```

---

### 🔀 when() / otherwise(): Lógica Condicional

**Sintaxis:**
```python
F.when(condicion, valor_si_true).otherwise(valor_si_false)
```

**Múltiples condiciones:**
```python
df = df.withColumn(
    "nivel_riesgo",
    F.when(F.col("score") >= 80, "Bajo")
     .when(F.col("score") >= 50, "Medio")
     .when(F.col("score") >= 20, "Alto")
     .otherwise("Crítico")
)
```

**Con AND / OR:**
```python
F.when(
    (F.col("ventas") > 100000) & (F.col("margen") > 0.20),
    "Premium"
).otherwise("Normal")
```

---

### 🚫 Manejo de Valores Nulos

**Detectar nulos:**
```python
df.filter(F.col("columna").isNull())
df.filter(F.col("columna").isNotNull())
```

**Reemplazar nulos:**
```python
# Método 1: fillna
df = df.fillna({"precio": 0, "nombre": "Sin nombre"})

# Método 2: coalesce (primera no nula)
df = df.withColumn(
    "precio_final",
    F.coalesce(F.col("precio_oferta"), F.col("precio_lista"), F.lit(0))
)
```

**Eliminar nulos:**
```python
df = df.dropna(subset=["columna_importante"])
```

---

### 🔄 Conversión de Tipos (Cast)

**Sintaxis:**
```python
df = df.withColumn("col_nueva", F.col("col_vieja").cast("tipo"))
```

**Tipos comunes:**
```python
# String → Integer
df = df.withColumn("edad", F.col("edad_str").cast("int"))

# String → Date
df = df.withColumn("fecha", F.to_date(F.col("fecha_str"), "yyyy-MM-dd"))

# Timestamp → Date
df = df.withColumn("solo_fecha", F.to_date(F.col("timestamp")))

# Double → Decimal
df = df.withColumn("precio", F.col("precio").cast(DecimalType(10, 2)))
```

---

### ⚠️ Errores Comunes

**❌ MAL: Usar Python directamente**
```python
# NO funciona distribuido
df = df.withColumn("doble", df["ventas"] * 2)  # Python
```

**✅ BIEN: Usar F.col()**
```python
# Ejecuta distribuido
df = df.withColumn("doble", F.col("ventas") * 2)  # Spark
```

---

### 📋 Caso de Uso: Enriquecimiento de Ventas

```python
from pyspark.sql import functions as F

# 1. Agregar columnas de fecha
df = df.withColumn("año", F.year(F.col("fecha")))
df = df.withColumn("mes", F.month(F.col("fecha")))
df = df.withColumn("trimestre", F.quarter(F.col("fecha")))

# 2. Calcular métricas
df = df.withColumn("margen", F.col("ventas") - F.col("costo"))
df = df.withColumn("margen_pct", F.col("margen") / F.col("ventas"))

# 3. Clasificar
df = df.withColumn(
    "segmento",
    F.when(F.col("ventas") > 100000, "A")
     .when(F.col("ventas") > 50000, "B")
     .otherwise("C")
)

# 4. Limpiar
df = df.fillna({"descuento": 0})
df = df.withColumn("nombre", F.upper(F.trim(F.col("nombre"))))
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("🔧 PYSPARK TRANSFORMACIONES AVANZADAS")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • withColumn() - Agregar/modificar columnas")
print("  • pyspark.sql.functions (F) - Biblioteca de funciones")
print("  • when/otherwise - Lógica condicional")
print("  • Manejo de nulos y conversiones de tipos")

print("\n📖 Funciones clave:")
print("  - df.withColumn('col', expresion)")
print("  - F.col('col'), F.lit(valor)")
print("  - F.when(cond, val).otherwise(val)")
print("  - F.isNull(), F.isNotNull()")
print("  - df.cast('tipo'), F.to_date()")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
print("Módulo 12: Transformaciones distribuidas avanzadas en PySpark")
print("Soporte para Joins distribuidos y agregaciones compuestas en Databricks.")



## 🎓 Conclusiones del notebook 12_01

### ✅ Lo que aprendiste

1. **withColumn() — agregar y modificar columnas:**
   - `df.withColumn('nueva', F.col('a') * 2)` — columna calculada
   - `df.withColumn('existente', F.upper(F.col('nombre')))` — sobrescribe
   - Inmutable: crea un NUEVO DataFrame, no modifica el original

2. **pyspark.sql.functions (F) — biblioteca esencial:**
   - Matemáticas: `F.round()`, `F.abs()`, `F.col() + F.lit(100)`
   - Strings: `F.upper()`, `F.trim()`, `F.concat()`, `F.substring()`
   - Fechas: `F.year()`, `F.month()`, `F.datediff()`, `F.to_date()`
   - Agregaciones: `F.sum()`, `F.avg()`, `F.count()`, `F.min()`, `F.max()`

3. **when() / otherwise() — lógica condicional:**
   - `F.when(cond, val).when(cond2, val2).otherwise(default)`
   - Equivalente a CASE WHEN de SQL
   - Soporta `&` (AND) y `|` (OR) para condiciones compuestas

4. **Manejo de nulos:**
   - `F.col('x').isNull()` / `.isNotNull()` — detectar
   - `df.fillna({'precio': 0})` — reemplazar con valor fijo
   - `F.coalesce(F.col('a'), F.col('b'), F.lit(0))` — primera no nula
   - `df.dropna(subset=['col'])` — eliminar filas con nulos

5. **Conversión de tipos (cast):**
   - `F.col('edad').cast('int')` — string a entero
   - `F.to_date(F.col('fecha_str'), 'yyyy-MM-dd')` — string a fecha
   - `F.col('precio').cast(DecimalType(10, 2))` — double a decimal
   - Producción: siempre validar tipos en carga (schema explícito)

---

### 🎯 Reglas de Oro

👉 **Regla #1: Usar F.col(), NO Python nativo**
```python
# MALO: Python nativo, no funciona distribuido
df = df.withColumn('doble', df['ventas'] * 2)
df = df.withColumn('mayus', df['nombre'].upper())

# BUENO: F.col() ejecuta distribuido en el cluster
df = df.withColumn('doble', F.col('ventas') * 2)
df = df.withColumn('mayus', F.upper(F.col('nombre')))
```

👉 **Regla #2: Encadenar withColumn para legibilidad**
```python
# MALO: una expresión gigante ilegible
df = df.withColumn('resultado', F.when((F.col('a') > 100) & (F.col('b') < 50), F.round(F.col('a') * 1.21, 2)).otherwise(F.col('b')))

# BUENO: encadenar pasos claros
df = (df
    .withColumn('iva', F.round(F.col('a') * 0.21, 2))
    .withColumn('condicion', (F.col('a') > 100) & (F.col('b') < 50))
    .withColumn('resultado', F.when(F.col('condicion'), F.col('a') + F.col('iva')).otherwise(F.col('b')))
    .drop('condicion')
)
```

👉 **Regla #3: coalesce antes que fillna para lógica de negocio**
```python
# MALO: fillna con un valor fijo puede ocultar problemas
df = df.fillna({'precio': 0})  # ¿0 es correcto? ¿Y si era un error?

# BUENO: coalesce con jerarquía de columnas
df = df.withColumn('precio_final',
    F.coalesce(F.col('precio_oferta'), F.col('precio_lista'), F.lit(0))
)
# Usa oferta si existe, si no lista, si no 0
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Agregar columna calculada | `df.withColumn('col', F.col('a') * F.col('b'))` |
| Sobrescribir columna existente | `df.withColumn('col', F.upper(F.col('col')))` |
| Clasificar por rangos | `F.when(...).when(...).otherwise(...)` |
| Detectar nulos | `F.col('x').isNull()` |
| Reemplazar nulos con valor | `df.fillna({'col': valor})` |
| Reemplazar nulos con jerarquía | `F.coalesce(F.col('a'), F.col('b'), F.lit(0))` |
| Eliminar filas con nulos | `df.dropna(subset=['col'])` |
| String → Integer | `F.col('x').cast('int')` |
| String → Date | `F.to_date(F.col('x'), 'yyyy-MM-dd')` |
| Double → Decimal | `F.col('x').cast(DecimalType(10, 2))` |
| Extraer año/mes | `F.year(F.col('fecha'))`, `F.month(F.col('fecha'))` |
| Concatenar strings | `F.concat(F.col('a'), F.lit('-'), F.col('b'))` |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔧 ¡PySpark Transformación Avanzada dominado!</h3>
  <p><i>"withColumn + F.functions = el 80% del ETL. Dominarlos es dominar Spark."</i></p>
</div>